### Agentic RAG with LangGraph

In [36]:
import os
from typing import List, TypedDict
from langgraph.graph import StateGraph, END, START
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings



In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
# Set Your API Keys
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initiaze Models
llm=ChatGroq(model="llama3-70b-8192", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9438.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
llm


ChatGroq(profile={'max_input_tokens': 8192, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001990EE1E850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001990EE1F4D0>, model_name='llama3-70b-8192', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

### State Definiton


In [27]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [28]:
### Sample docs and Vector Store
sample_texts=[
    "The capital of France is Paris.",
    "The largest mammal is the blue whale.",
    "The Great Wall of China is visible from space."]

documents=[Document(page_content=text) for i, text in enumerate(sample_texts)]

### Create Vector Store

vectorstore = FAISS.from_documents(documents, embeddings)
retriever=vectorstore.as_retriever(k=2)


### Agents Function

In [29]:
def decide_retrieval(state: AgentState) -> AgentState:
    """Decide whether retrieval is needed based on the question."""
    question=state["question"]
    
    # Simple heuristic: if the question contains "capital", "largest", or "Great Wall", we need retrieval
    keywords = ["capital", "largest", "Great Wall"]
    return any(keyword in state["question"] for keyword in keywords)

In [30]:
### Retrive docs
def retrieve_docs(state: AgentState) -> AgentState:
    """Retrieve relevant documents based on the question.
    """
    question=state["question"]
    documents=retriever.invoke(question)
    
    state["documents"] = documents
    return state

In [31]:
### generate answer
def generate_answer(state: AgentState) -> AgentState:
    """Generate an answer based on the question and retrieved documents."""
    question=state["question"]
    documents=state.get("documents", [])
    
    
    if documents:
        # RAG Approach: Use retrieved documents to generate answer
        context="\n".join([doc.page_content for doc in documents])
        prompt=f"""Question: {question}\nContext: {context}\nAnswer:
        
    Context:
    {context}
    
    Question:{question}
    
    Answer:"""
        
    else:
        # Direct LLM Approach: Generate answer without retrieval
        prompt=f"""Question: {question}\nAnswer:"""
    
    response=llm.invoke(prompt)
    answer=response.content
    
    state["answer"] = answer
    return state
    
    

### Conditional logic

In [ ]:
def should_retrieve(state: AgentState) -> str:
    if state["needs_retrieval"]:
        return "retrieve_docs" # Matches your mapping key
    else:
        return "generate_answer" # Matches your mapping key

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


ValueError: Branch with name `should_retrieve` already exists for node `decide`

### Build the Graph


In [47]:
# Create the state graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("decide", decide_retrieval)
workflow.add_node("retrieve", retrieve_docs)
workflow.add_node("generate", generate_answer)

# Set Entry Point by START edge
workflow.add_edge(START, "decide")

# Add Conditional edges
workflow.add_conditional_edges(
    "decide",
    should_retrieve,
    {
        "retrieve_docs": "retrieve",
        "generate_answer": "generate"
    }
)

# Add Edges
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

### Test the Agentic System

In [48]:
# --- STEP 1: Define the State ---
class AgentState(TypedDict):
    question: str
    documents: List[str]  # Make sure this name matches everywhere!
    answer: str
    needs_retrieval: bool

# --- STEP 2: Compile the Graph ---
# This MUST happen before you call ask_question
app = workflow.compile()

# --- STEP 3: The Helper Function ---
def ask_question(question: str):
    """Helper Function to ask a question and get an answer."""
    
    # This dictionary must match the keys in AgentState
    initial_state = {
        "question": question,
        "documents": [],
        "answer": "",
        "needs_retrieval": False
    }
    
    # invoke runs the graph from START to END
    result = app.invoke(initial_state)
    
    # We return the "answer" key which should be populated by your 'generate' node
    return result["answer"]

# --- STEP 4: Test it! ---
final_answer = ask_question("What is the capital of France?")
print(f"Final RAG Answer: {final_answer}")

InvalidUpdateError: Expected dict, got True
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE